# 02 Cleaning - UK Road Accident Data

Raw input: data/raw/UK_Accident.csv
Clean output: data/processed/UK_Accident_cleaned.csv
Transformation log: data/processed/cleaning_transformation_log.csv

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, UTC

ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / 'data' / 'raw' / 'UK_Accident.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PATH = PROCESSED_DIR / 'UK_Accident_cleaned.csv'
LOG_PATH = PROCESSED_DIR / 'cleaning_transformation_log.csv'

df = pd.read_csv(RAW_PATH, low_memory=False)
transformation_log = []

def capture_state(frame):
    return frame.shape[0], frame.shape[1], int(frame.isna().sum().sum())

def log_step(step, rows_before, rows_after, cols_before, cols_after, null_before, null_after, note=''):
    transformation_log.append({
        'timestamp_utc': datetime.now(UTC).isoformat(),
        'step': step,
        'rows_before': int(rows_before),
        'rows_after': int(rows_after),
        'cols_before': int(cols_before),
        'cols_after': int(cols_after),
        'null_before': int(null_before),
        'null_after': int(null_after),
        'rows_changed': int(rows_before - rows_after),
        'note': note
    })

rows_b, cols_b, null_b = capture_state(df)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')
if 'Unnamed:_0' in df.columns:
    df = df.drop(columns=['Unnamed:_0'])
rows_a, cols_a, null_a = capture_state(df)
log_step('standardize_column_names_and_drop_index_column', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Dropped auto index column if present')

rows_b, cols_b, null_b = capture_state(df)
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].astype(str).str.strip()
missing_tokens = {'', 'NA', 'N/A', 'NULL', 'NONE', 'nan'}
for col in df.select_dtypes(include=['object', 'string']).columns:
    upper_series = df[col].str.upper()
    df.loc[upper_series.isin(missing_tokens), col] = np.nan
rows_a, cols_a, null_a = capture_state(df)
log_step('trim_text_and_mark_common_missing_tokens', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Normalized object columns and missing markers')

rows_b, cols_b, null_b = capture_state(df)
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
if 'Time' in df.columns:
    parsed_time = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce')
    df['Time'] = parsed_time.dt.time
numeric_cols = ['Location_Easting_OSGR', 'Location_Northing_OSGR', 'Longitude', 'Latitude', 'Police_Force', 'Accident_Severity', 'Number_of_Vehicles', 'Number_of_Casualties', 'Day_of_Week', 'Local_Authority_(District)', '1st_Road_Class', '1st_Road_Number', 'Speed_limit', '2nd_Road_Class', '2nd_Road_Number', 'Weather_Conditions', 'Urban_or_Rural_Area', 'Year']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
rows_a, cols_a, null_a = capture_state(df)
log_step('type_conversion_dates_times_numeric', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Coerced invalid values to NaN/NaT')

rows_b, cols_b, null_b = capture_state(df)
if 'Accident_Index' in df.columns:
    df = df.drop_duplicates(subset=['Accident_Index'])
df = df.drop_duplicates()
rows_a, cols_a, null_a = capture_state(df)
log_step('remove_duplicates', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Removed duplicate Accident_Index and exact duplicate rows')

rows_b, cols_b, null_b = capture_state(df)
if 'Longitude' in df.columns:
    df.loc[~df['Longitude'].between(-8, 2), 'Longitude'] = np.nan
if 'Latitude' in df.columns:
    df.loc[~df['Latitude'].between(49, 61), 'Latitude'] = np.nan
if 'Speed_limit' in df.columns:
    df.loc[~df['Speed_limit'].between(10, 80), 'Speed_limit'] = np.nan
if 'Year' in df.columns:
    df.loc[~df['Year'].between(2000, 2018), 'Year'] = np.nan
if 'Accident_Severity' in df.columns:
    df.loc[~df['Accident_Severity'].between(1, 3), 'Accident_Severity'] = np.nan
rows_a, cols_a, null_a = capture_state(df)
log_step('range_validation_to_nan', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Applied practical UK domain ranges')

rows_b, cols_b, null_b = capture_state(df)
critical_cols = [col for col in ['Accident_Index', 'Date', 'Accident_Severity', 'Year'] if col in df.columns]
df = df.dropna(subset=critical_cols)
if 'Date' in df.columns:
    df['Month'] = df['Date'].dt.month
    df['Day'] = df['Date'].dt.day
rows_a, cols_a, null_a = capture_state(df)
log_step('drop_critical_nulls_and_add_date_parts', rows_b, rows_a, cols_b, cols_a, null_b, null_a, 'Removed rows missing critical fields and added Month/Day')

if 'Date' in df.columns and 'Time' in df.columns:
    df = df.sort_values(by=['Date', 'Time'], kind='stable').reset_index(drop=True)
elif 'Date' in df.columns:
    df = df.sort_values(by=['Date'], kind='stable').reset_index(drop=True)

df.to_csv(CLEAN_PATH, index=False)
pd.DataFrame(transformation_log).to_csv(LOG_PATH, index=False)
summary = pd.DataFrame(transformation_log)
print('Final cleaned shape:', df.shape)
summary